In [ ]:
%%shell

wget --keep-session-cookies --save-cookies=cookies.txt --post-data 'username=villegas.samuel%40titans.easternflorida.edu&password=EBF5m%26n%5EzxbPWC&submit=Login' https://www.cityscapes-dataset.com/login/
wget --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=3
wget --load-cookies cookies.txt --content-disposition https://www.cityscapes-dataset.com/file-handling/?packageID=1

--2025-12-13 02:40:56--  https://www.cityscapes-dataset.com/login/
Resolving www.cityscapes-dataset.com (www.cityscapes-dataset.com)... 139.19.217.8
Connecting to www.cityscapes-dataset.com (www.cityscapes-dataset.com)|139.19.217.8|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.cityscapes-dataset.com/downloads/ [following]
--2025-12-13 02:40:58--  https://www.cityscapes-dataset.com/downloads/
Reusing existing connection to www.cityscapes-dataset.com:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘index.html.1’

index.html.1            [  <=>               ]  55.82K   216KB/s    in 0.3s    

2025-12-13 02:40:58 (216 KB/s) - ‘index.html.1’ saved [57162]

zsh:1: no matches found: https://www.cityscapes-dataset.com/file-handling/?packageID=3
zsh:1: no matches found: https://www.cityscapes-dataset.com/file-handling/?packageID=1


In [ ]:
%%shell

mkdir datasets/
mkdir datasets/cityscapes

mkdir datasets/cityscapes/leftImg8bit/
mkdir datasets/cityscapes/gtFine/

unzip -o leftImg8bit_trainvaltest.zip
unzip -o gtFine_trainvaltest.zip

cp -r leftImg8bit/** datasets/cityscapes/leftImg8bit
cp -r gtFine/** datasets/cityscapes/gtFine

# rm -rf datasets/cityscapes/gtFine_trainvaltest.zip datasets/cityscapes/leftImg8bit_trainvaltest.zip

Archive:  leftImg8bit_trainvaltest.zip
 extracting: README                  
 extracting: license.txt             
   creating: leftImg8bit/train/
   creating: leftImg8bit/train/jena/
 extracting: leftImg8bit/train/jena/jena_000078_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000032_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000055_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000067_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000001_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000111_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000114_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000105_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000021_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000045_000019_leftImg8bit.png  
 extracting: leftImg8bit/train/jena/jena_000058_000019_leftImg8bit.png  
 extracting: 

In [1]:
!ls datasets/cityscapes

ls: cannot access 'datasets/cityscapes': No such file or directory


In [4]:
from torchvision.datasets import Cityscapes
import os
import argparse
import time
import shutil

import torch
import torch.utils.data as data
import torch.backends.cudnn as cudnn

from torchvision.transforms import v2 as transforms

In [5]:
###########################################################################
# Created by: Tramac
# Date: 2019-03-25
# Copyright (c) 2017
###########################################################################

"""Fast Segmentation Convolutional Neural Network"""
import os
import struct
from typing import List, Tuple
import torch
import torch.jit as jit
import torch.nn as nn
import torch.nn.functional as F

__all__ = ["FastSCNN", "get_fast_scnn"]


class FastSCNN(nn.Module):
    def __init__(self, num_classes: int, aux=False, **kwargs):
        super(FastSCNN, self).__init__()
        self.num_classes = num_classes
        self.aux = aux
        self.learning_to_downsample = LearningToDownsample(32, 48, 64)
        self.global_feature_extractor = GlobalFeatureExtractor(
            64, [64, 96, 128], 128, 6, [3, 3, 3]
        )
        self.feature_fusion = FeatureFusionModule(64, 128, 128)
        self.classifier = Classifer(128, self.num_classes)
        if self.aux:
            self.auxlayer = nn.Sequential(
                nn.Conv2d(64, 32, 3, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(True),
                nn.Dropout(0.1),
                nn.Conv2d(32, self.num_classes, 1),
            )

    def forward(self, x):
        size = x.size()[2:]
        # X Must be converted from float to byte
        higher_res_features = self.learning_to_downsample(x)
        x = self.global_feature_extractor(higher_res_features)
        x = self.feature_fusion(higher_res_features, x)
        x = self.classifier(x)
        outputs: list[torch.Tensor] = []
        x = F.interpolate(x, size, mode="bilinear", align_corners=True)
        # outputs.append(x)
        # if self.aux:
        #     auxout = self.auxlayer(higher_res_features)
        #     auxout = F.interpolate(auxout, size, mode="bilinear", align_corners=True)
        #     outputs.append(auxout)
        # return tuple(outputs)

        # there was a check to make sure aux wasnt null but it was removed, add a torchscript version for that condition please future me.
        if hasattr(self, "auxlayer"):
            auxout = self.auxlayer(higher_res_features)
            auxout = F.interpolate(auxout, size, mode="bilinear", align_corners=True)
        else:
            auxout = None
        return x, auxout


class _ConvBNReLU(nn.Module):
    """Conv-BN-ReLU"""

    def __init__(
        self, in_channels, out_channels, kernel_size=3, stride=1, padding=0, **kwargs
    ):
        super(_ConvBNReLU, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels, out_channels, kernel_size, stride, padding, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.conv(x)


class _DSConv(nn.Module):
    """Depthwise Separable Convolutions"""

    def __init__(self, dw_channels, out_channels, stride=1, **kwargs):
        super(_DSConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(
                dw_channels, dw_channels, 3, stride, 1, groups=dw_channels, bias=False
            ),
            nn.BatchNorm2d(dw_channels),
            nn.ReLU(True),
            nn.Conv2d(dw_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.conv(x)


class _DWConv(nn.Module):
    def __init__(self, dw_channels, out_channels, stride=1, **kwargs):
        super(_DWConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(
                dw_channels, out_channels, 3, stride, 1, groups=dw_channels, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.conv(x)


class LinearBottleneck(nn.Module):
    """LinearBottleneck used in MobileNetV2"""

    def __init__(self, in_channels, out_channels, t=6, stride=2, **kwargs):
        super(LinearBottleneck, self).__init__()
        self.use_shortcut = stride == 1 and in_channels == out_channels
        self.block = nn.Sequential(
            # pw
            _ConvBNReLU(in_channels, in_channels * t, 1),
            # dw
            _DWConv(in_channels * t, in_channels * t, stride),
            # pw-linear
            nn.Conv2d(in_channels * t, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        out = self.block(x)
        if self.use_shortcut:
            out = x + out
        return out


class PyramidPooling(nn.Module):
    """Pyramid pooling module"""

    def __init__(self, in_channels, out_channels, **kwargs):
        super(PyramidPooling, self).__init__()
        inter_channels = int(in_channels / 4)
        self.conv1 = _ConvBNReLU(in_channels, inter_channels, 1, **kwargs)
        self.conv2 = _ConvBNReLU(in_channels, inter_channels, 1, **kwargs)
        self.conv3 = _ConvBNReLU(in_channels, inter_channels, 1, **kwargs)
        self.conv4 = _ConvBNReLU(in_channels, inter_channels, 1, **kwargs)
        self.out = _ConvBNReLU(in_channels * 2, out_channels, 1)
        self.pool1 = nn.AdaptiveAvgPool2d(1)
        self.pool2 = nn.AdaptiveAvgPool2d(2)
        self.pool3 = nn.AdaptiveAvgPool2d(3)
        self.pool6 = nn.AdaptiveAvgPool2d(6)

    def pool(self, x, size):
        avgpool = nn.AdaptiveAvgPool2d(size)
        return avgpool(x)

    def upsample(self, x, size: List[int]) -> torch.Tensor:
        return F.interpolate(x, size, mode="bilinear", align_corners=True)

    def forward(self, x):
        size = x.size()[2:]
        feat1 = self.upsample(self.conv1(self.pool1(x)), size)
        feat2 = self.upsample(self.conv2(self.pool2(x)), size)
        feat3 = self.upsample(self.conv3(self.pool3(x)), size)
        feat4 = self.upsample(self.conv4(self.pool6(x)), size)
        x = torch.cat([x, feat1, feat2, feat3, feat4], dim=1)
        x = self.out(x)
        return x


class LearningToDownsample(nn.Module):
    """Learning to downsample module"""

    def __init__(self, dw_channels1=32, dw_channels2=48, out_channels=64, **kwargs):
        super(LearningToDownsample, self).__init__()
        self.conv = _ConvBNReLU(3, dw_channels1, 3, stride=2)
        self.dsconv1 = _DSConv(dw_channels1, dw_channels2, 2)
        self.dsconv2 = _DSConv(dw_channels2, out_channels, 2)

    def forward(self, x):
        x = self.conv(x)
        x = self.dsconv1(x)
        x = self.dsconv2(x)
        return x


class GlobalFeatureExtractor(nn.Module):
    """Global feature extractor module"""

    def __init__(
        self,
        in_channels=64,
        block_channels=(64, 96, 128),
        out_channels=128,
        t=6,
        num_blocks=(3, 3, 3),
        **kwargs
    ):
        super(GlobalFeatureExtractor, self).__init__()
        self.bottleneck1 = self._make_layer(
            LinearBottleneck, in_channels, block_channels[0], num_blocks[0], t, 2
        )
        self.bottleneck2 = self._make_layer(
            LinearBottleneck, block_channels[0], block_channels[1], num_blocks[1], t, 2
        )
        self.bottleneck3 = self._make_layer(
            LinearBottleneck, block_channels[1], block_channels[2], num_blocks[2], t, 1
        )
        self.ppm = PyramidPooling(block_channels[2], out_channels)

    def _make_layer(self, block, inplanes, planes, blocks, t=6, stride=1):
        layers = []
        layers.append(block(inplanes, planes, t, stride))
        for i in range(1, blocks):
            layers.append(block(planes, planes, t, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.bottleneck1(x)
        x = self.bottleneck2(x)
        x = self.bottleneck3(x)
        x = self.ppm(x)
        return x


class FeatureFusionModule(nn.Module):
    """Feature fusion module"""

    def __init__(
        self,
        highter_in_channels,
        lower_in_channels,
        out_channels,
        scale_factor=4,
        **kwargs
    ):
        super(FeatureFusionModule, self).__init__()
        self.scale_factor = scale_factor
        self.dwconv = _DWConv(lower_in_channels, out_channels, 1)
        self.conv_lower_res = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1), nn.BatchNorm2d(out_channels)
        )
        self.conv_higher_res = nn.Sequential(
            nn.Conv2d(highter_in_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels),
        )
        self.relu = nn.ReLU(True)

    def forward(self, higher_res_feature, lower_res_feature):
        lower_res_feature = F.interpolate(
            lower_res_feature, scale_factor=4.0, mode="bilinear", align_corners=True
        )
        lower_res_feature = self.dwconv(lower_res_feature)
        lower_res_feature = self.conv_lower_res(lower_res_feature)
        higher_res_feature = self.conv_higher_res(higher_res_feature)

        # Resizing the lower resulution tensor to the size of the higher resolution tensor. resize_ is unsupported in torchscript
        # out = higher_res_feature + lower_res_feature.resize_(higher_res_feature.size())
        out = higher_res_feature + F.interpolate(
            lower_res_feature,
            size=higher_res_feature.shape[2:],
            mode="bilinear",
            align_corners=True,
        )
        return self.relu(out)


class Classifer(nn.Module):
    """Classifer"""

    def __init__(self, dw_channels, num_classes, stride=1, **kwargs):
        super(Classifer, self).__init__()
        self.dsconv1 = _DSConv(dw_channels, dw_channels, stride)
        self.dsconv2 = _DSConv(dw_channels, dw_channels, stride)
        self.conv = nn.Sequential(
            nn.Dropout(0.1), nn.Conv2d(dw_channels, num_classes, 1)
        )

    def forward(self, x):
        x = self.dsconv1(x)
        x = self.dsconv2(x)
        x = self.conv(x)
        return x


def get_fast_scnn(
    dataset="citys", pretrained=False, root="./weights", map_cpu=False, **kwargs
):
    acronyms = {
        "pascal_voc": "voc",
        "pascal_aug": "voc",
        "ade20k": "ade",
        "coco": "coco",
        "citys": "citys",
    }

    # 19 is the total number of classes in the citys dataset
    model = FastSCNN(19, **kwargs)
    if pretrained:
        if map_cpu:
            model.load_state_dict(
                torch.load(
                    os.path.join(root, "fast_scnn_%s.pth" % acronyms[dataset]),
                    map_location="cpu",
                )
            )
        else:
            model.load_state_dict(
                torch.load(os.path.join(root, "fast_scnn_%s.pth" % acronyms[dataset]))
            )
    return model


if __name__ == "__main__":
    img = torch.randn(2, 3, 256, 512)
    model = get_fast_scnn("citys")
    outputs = model(img)

In [6]:
"""Popular Learning Rate Schedulers"""

from __future__ import division
import math


class LRScheduler(object):
    r"""Learning Rate Scheduler

    Parameters
    ----------
    mode : str
        Modes for learning rate scheduler.
        Currently it supports 'constant', 'step', 'linear', 'poly' and 'cosine'.
    base_lr : float
        Base learning rate, i.e. the starting learning rate.
    target_lr : float
        Target learning rate, i.e. the ending learning rate.
        With constant mode target_lr is ignored.
    niters : int
        Number of iterations to be scheduled.
    nepochs : int
        Number of epochs to be scheduled.
    iters_per_epoch : int
        Number of iterations in each epoch.
    offset : int
        Number of iterations before this scheduler.
    power : float
        Power parameter of poly scheduler.
    step_iter : list
        A list of iterations to decay the learning rate.
    step_epoch : list
        A list of epochs to decay the learning rate.
    step_factor : float
        Learning rate decay factor.
    """

    def __init__(
        self,
        mode,
        base_lr=0.01,
        target_lr=0,
        niters=0,
        nepochs=0,
        iters_per_epoch=0,
        offset=0,
        power=2.0,
        step_iter=None,
        step_epoch=None,
        step_factor=0.1,
    ):
        super(LRScheduler, self).__init__()
        assert mode in ["constant", "step", "linear", "poly", "cosine"]

        self.mode = mode
        if mode == "step":
            assert step_iter is not None or step_epoch is not None
        self.base_lr = base_lr
        self.target_lr = target_lr
        if self.mode == "constant":
            self.target_lr = self.base_lr

        self.niters = niters
        self.step = step_iter
        epoch_iters = nepochs * iters_per_epoch
        if epoch_iters > 0:
            self.niters = epoch_iters
            if step_epoch is not None:
                self.step = [s * iters_per_epoch for s in step_epoch]

        self.offset = offset
        self.power = power
        self.step_factor = step_factor

    def __call__(self, num_update):
        self.update(num_update)
        return self.learning_rate

    def update(self, num_update):
        N = self.niters - 1
        T = num_update - self.offset
        T = min(max(0, T), N)

        if self.mode == "constant":
            factor = 0
        elif self.mode == "linear":
            factor = 1 - T / N
        elif self.mode == "poly":
            factor = pow(1 - T / N, self.power)
        elif self.mode == "cosine":
            factor = (1 + math.cos(math.pi * T / N)) / 2
        elif self.mode == "step":
            if self.step is not None:
                count = sum([1 for s in self.step if s <= T])
                factor = pow(self.step_factor, count)
            else:
                factor = 1
        else:
            raise NotImplementedError

        if self.mode == "step":
            self.learning_rate = self.base_lr * factor
        else:
            self.learning_rate = (
                self.target_lr + (self.base_lr - self.target_lr) * factor
            )


if __name__ == "__main__":
    lr_scheduler = LRScheduler(
        mode="poly", base_lr=0.01, nepochs=60, iters_per_epoch=176, power=0.9
    )
    for i in range(60 * 176):
        lr = lr_scheduler(i)
        print(lr)

0.01
0.009999147642521152
0.00999829527696917
0.009997442903343213
0.009996590521642439
0.009995738131866007
0.009994885734013075
0.0099940333280828
0.00999318091407434
0.009992328491986856
0.009991476061819504
0.009990623623571438
0.009989771177241822
0.009988918722829806
0.009988066260334551
0.009987213789755214
0.00998636131109095
0.009985508824340917
0.00998465632950427
0.009983803826580168
0.009982951315567761
0.00998209879646621
0.00998124626927467
0.009980393733992295
0.009979541190618238
0.00997868863915166
0.00997783607959171
0.009976983511937548
0.009976130936188326
0.009975278352343199
0.009974425760401319
0.009973573160361843
0.009972720552223924
0.009971867935986714
0.009971015311649371
0.009970162679211043
0.009969310038670888
0.009968457390028057
0.0099676047332817
0.009966752068430977
0.009965899395475033
0.009965046714413025
0.009964194025244103
0.009963341327967421
0.00996248862258213
0.009961635909087382
0.009960783187482327
0.009959930457766117
0.009959077719937906


In [7]:
"""Custom losses."""

import torch
import torch.nn as nn
import numpy as np

from torch.autograd import Variable

__all__ = ["MixSoftmaxCrossEntropyLoss", "MixSoftmaxCrossEntropyOHEMLoss"]


class MixSoftmaxCrossEntropyLoss(nn.CrossEntropyLoss):
    def __init__(self, aux=True, aux_weight=0.2, ignore_label=-1, **kwargs):
        super(MixSoftmaxCrossEntropyLoss, self).__init__(ignore_index=ignore_label)
        self.aux = aux
        self.aux_weight = aux_weight

    def _aux_forward(self, *inputs, **kwargs):
        *preds, target = tuple(inputs)

        loss = super(MixSoftmaxCrossEntropyLoss, self).forward(preds[0], target)
        for i in range(1, len(preds)):
            aux_loss = super(MixSoftmaxCrossEntropyLoss, self).forward(preds[i], target)
            loss += self.aux_weight * aux_loss
        return loss

    def forward(self, *inputs, **kwargs):
        preds, target = tuple(inputs)
        inputs = tuple(list(preds) + [target])
        if self.aux:
            return self._aux_forward(*inputs)
        else:
            return super(MixSoftmaxCrossEntropyLoss, self).forward(*inputs)


class SoftmaxCrossEntropyOHEMLoss(nn.Module):
    def __init__(
        self, ignore_label=-1, thresh=0.7, min_kept=256, use_weight=True, **kwargs
    ):
        super(SoftmaxCrossEntropyOHEMLoss, self).__init__()
        self.ignore_label = ignore_label
        self.thresh = float(thresh)
        self.min_kept = int(min_kept)
        if use_weight:
            print("w/ class balance")
            weight = torch.FloatTensor(
                [
                    0.8373,
                    0.918,
                    0.866,
                    1.0345,
                    1.0166,
                    0.9969,
                    0.9754,
                    1.0489,
                    0.8786,
                    1.0023,
                    0.9539,
                    0.9843,
                    1.1116,
                    0.9037,
                    1.0865,
                    1.0955,
                    1.0865,
                    1.1529,
                    1.0507,
                ]
            )
            self.criterion = torch.nn.CrossEntropyLoss(
                weight=weight, ignore_index=ignore_label
            )
        else:
            print("w/o class balance")
            self.criterion = torch.nn.CrossEntropyLoss(ignore_index=ignore_label)

    def forward(self, predict, target, weight=None):
        assert not target.requires_grad
        assert predict.dim() == 4
        assert target.dim() == 3
        assert predict.size(0) == target.size(0), "{0} vs {1} ".format(
            predict.size(0), target.size(0)
        )
        assert predict.size(2) == target.size(1), "{0} vs {1} ".format(
            predict.size(2), target.size(1)
        )
        assert predict.size(3) == target.size(2), "{0} vs {1} ".format(
            predict.size(3), target.size(3)
        )

        n, c, h, w = predict.size()
        input_label = target.data.cpu().numpy().ravel().astype(np.int32)
        x = np.rollaxis(predict.data.cpu().numpy(), 1).reshape((c, -1))
        input_prob = np.exp(x - x.max(axis=0).reshape((1, -1)))
        input_prob /= input_prob.sum(axis=0).reshape((1, -1))

        valid_flag = input_label != self.ignore_label
        valid_inds = np.where(valid_flag)[0]
        label = input_label[valid_flag]
        num_valid = valid_flag.sum()
        if self.min_kept >= num_valid:
            print("Labels: {}".format(num_valid))
        elif num_valid > 0:
            prob = input_prob[:, valid_flag]
            pred = prob[label, np.arange(len(label), dtype=np.int32)]
            threshold = self.thresh
            if self.min_kept > 0:
                index = pred.argsort()
                threshold_index = index[min(len(index), self.min_kept) - 1]
                if pred[threshold_index] > self.thresh:
                    threshold = pred[threshold_index]
            kept_flag = pred <= threshold
            valid_inds = valid_inds[kept_flag]

        label = input_label[valid_inds].copy()
        input_label.fill(self.ignore_label)
        input_label[valid_inds] = label
        valid_flag_new = input_label != self.ignore_label
        # print(np.sum(valid_flag_new))
        target = Variable(
            torch.from_numpy(input_label.reshape(target.size())).long().cuda()
        )

        return self.criterion(predict, target)


class MixSoftmaxCrossEntropyOHEMLoss(SoftmaxCrossEntropyOHEMLoss):
    def __init__(self, aux=False, aux_weight=0.2, ignore_index=-1, **kwargs):
        super(MixSoftmaxCrossEntropyOHEMLoss, self).__init__(
            ignore_label=ignore_index, **kwargs
        )
        self.aux = aux
        self.aux_weight = aux_weight

    def _aux_forward(self, *inputs, **kwargs):
        *preds, target = tuple(inputs)

        loss = super(MixSoftmaxCrossEntropyOHEMLoss, self).forward(preds[0], target)
        for i in range(1, len(preds)):
            aux_loss = super(MixSoftmaxCrossEntropyOHEMLoss, self).forward(
                preds[i], target
            )
            loss += self.aux_weight * aux_loss
        return loss

    def forward(self, *inputs, **kwargs):
        preds, target = tuple(inputs)
        inputs = tuple(list(preds) + [target])
        if self.aux:
            return self._aux_forward(*inputs)
        else:
            return super(MixSoftmaxCrossEntropyOHEMLoss, self).forward(*inputs)

In [8]:
from __future__ import division

import threading
import numpy as np

__all__ = [
    "SegmentationMetric",
    "batch_pix_accuracy",
    "batch_intersection_union",
    "pixelAccuracy",
    "intersectionAndUnion",
    "hist_info",
    "compute_score",
]

"""Evaluation Metrics for Semantic Segmentation"""


class SegmentationMetric(object):
    """Computes pixAcc and mIoU metric scores"""

    def __init__(self, nclass):
        super(SegmentationMetric, self).__init__()
        self.nclass = nclass
        self.lock = threading.Lock()
        self.reset()

    def update(self, preds, labels):
        """Updates the internal evaluation result.

        Parameters
        ----------
        labels : 'NumpyArray' or list of `NumpyArray`
            The labels of the data.
        preds : 'NumpyArray' or list of `NumpyArray`
            Predicted values.
        """
        if isinstance(preds, np.ndarray):
            self.evaluate_worker(preds, labels)
        elif isinstance(preds, (list, tuple)):
            threads = [
                threading.Thread(
                    target=self.evaluate_worker,
                    args=(pred, label),
                )
                for (pred, label) in zip(preds, labels)
            ]
            for thread in threads:
                thread.start()
            for thread in threads:
                thread.join()

    def get(self):
        """Gets the current evaluation result.

        Returns
        -------
        metrics : tuple of float
            pixAcc and mIoU
        """
        pixAcc = 1.0 * self.total_correct / (np.spacing(1) + self.total_label)
        IoU = 1.0 * self.total_inter / (np.spacing(1) + self.total_union)
        # It has same result with np.nanmean() when all class exist
        mIoU = IoU.mean()
        return pixAcc, mIoU

    def evaluate_worker(self, pred, label):
        correct, labeled = batch_pix_accuracy(pred, label)
        inter, union = batch_intersection_union(pred, label, self.nclass)
        with self.lock:
            self.total_correct += correct
            self.total_label += labeled
            self.total_inter += inter
            self.total_union += union

    def reset(self):
        """Resets the internal evaluation result to initial state."""
        self.total_inter = 0
        self.total_union = 0
        self.total_correct = 0
        self.total_label = 0


def batch_pix_accuracy(predict, target):
    """PixAcc"""
    # inputs are numpy array, output 4D, target 3D
    assert predict.shape == target.shape
    predict = predict.astype("int64") + 1
    target = target.astype("int64") + 1

    pixel_labeled = np.sum(target > 0)
    pixel_correct = np.sum((predict == target) * (target > 0))
    assert pixel_correct <= pixel_labeled, "Correct area should be smaller than Labeled"
    return pixel_correct, pixel_labeled


def batch_intersection_union(predict, target, nclass):
    """mIoU"""
    # inputs are numpy array, output 4D, target 3D
    assert predict.shape == target.shape
    mini = 1
    maxi = nclass
    nbins = nclass
    predict = predict.astype("int64") + 1
    target = target.astype("int64") + 1

    predict = predict * (target > 0).astype(predict.dtype)
    intersection = predict * (predict == target)
    # areas of intersection and union
    # element 0 in intersection occur the main difference from np.bincount. set boundary to -1 is necessary.
    area_inter, _ = np.histogram(intersection, bins=nbins, range=(mini, maxi))
    area_pred, _ = np.histogram(predict, bins=nbins, range=(mini, maxi))
    area_lab, _ = np.histogram(target, bins=nbins, range=(mini, maxi))
    area_union = area_pred + area_lab - area_inter
    assert (
        area_inter <= area_union
    ).all(), "Intersection area should be smaller than Union area"
    return area_inter, area_union


def pixelAccuracy(imPred, imLab):
    """
    This function takes the prediction and label of a single image, returns pixel-wise accuracy
    To compute over many images do:
    for i = range(Nimages):
         (pixel_accuracy[i], pixel_correct[i], pixel_labeled[i]) = \
            pixelAccuracy(imPred[i], imLab[i])
    mean_pixel_accuracy = 1.0 * np.sum(pixel_correct) / (np.spacing(1) + np.sum(pixel_labeled))
    """
    # Remove classes from unlabeled pixels in gt image.
    # We should not penalize detections in unlabeled portions of the image.
    pixel_labeled = np.sum(imLab >= 0)
    pixel_correct = np.sum((imPred == imLab) * (imLab >= 0))
    pixel_accuracy = 1.0 * pixel_correct / pixel_labeled
    return (pixel_accuracy, pixel_correct, pixel_labeled)


def intersectionAndUnion(imPred, imLab, numClass):
    """
    This function takes the prediction and label of a single image,
    returns intersection and union areas for each class
    To compute over many images do:
    for i in range(Nimages):
        (area_intersection[:,i], area_union[:,i]) = intersectionAndUnion(imPred[i], imLab[i])
    IoU = 1.0 * np.sum(area_intersection, axis=1) / np.sum(np.spacing(1)+area_union, axis=1)
    """
    # Remove classes from unlabeled pixels in gt image.
    # We should not penalize detections in unlabeled portions of the image.
    imPred = imPred * (imLab >= 0)

    # Compute area intersection:
    intersection = imPred * (imPred == imLab)
    (area_intersection, _) = np.histogram(
        intersection, bins=numClass, range=(1, numClass)
    )

    # Compute area union:
    (area_pred, _) = np.histogram(imPred, bins=numClass, range=(1, numClass))
    (area_lab, _) = np.histogram(imLab, bins=numClass, range=(1, numClass))
    area_union = area_pred + area_lab - area_intersection
    return (area_intersection, area_union)


def hist_info(pred, label, num_cls):
    assert pred.shape == label.shape
    k = (label >= 0) & (label < num_cls)
    labeled = np.sum(k)
    correct = np.sum((pred[k] == label[k]))

    return (
        np.bincount(
            num_cls * label[k].astype(int) + pred[k], minlength=num_cls**2
        ).reshape(num_cls, num_cls),
        labeled,
        correct,
    )


def compute_score(hist, correct, labeled):
    iu = np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist))
    # print('right')
    # print(iu)
    mean_IU = np.nanmean(iu)
    mean_IU_no_back = np.nanmean(iu[1:])
    freq = hist.sum(1) / hist.sum()
    freq_IU = (iu[freq > 0] * freq[freq > 0]).sum()
    mean_pixel_acc = correct / labeled

    return iu, mean_IU, mean_IU_no_back, mean_pixel_acc

In [9]:
"""Visualization Utils"""

from PIL import Image

__all__ = ["get_color_pallete"]


def get_color_pallete(npimg, dataset="citys"):
    """Visualize image.

    Parameters
    ----------
    npimg : numpy.ndarray
        Single channel image with shape `H, W, 1`.
    dataset : str, default: 'pascal_voc'
        The dataset that model pretrained on. ('pascal_voc', 'ade20k')
    Returns
    -------
    out_img : PIL.Image
        Image with color pallete
    """
    # recovery boundary
    if dataset in ("pascal_voc", "pascal_aug"):
        npimg[npimg == -1] = 255
    # put colormap
    if dataset == "ade20k":
        npimg = npimg + 1
        out_img = Image.fromarray(npimg.astype("uint8"))
        out_img.putpalette(adepallete)
        return out_img
    elif dataset == "citys":
        out_img = Image.fromarray(npimg.astype("uint8"))
        out_img.putpalette(cityspallete)
        return out_img
    out_img = Image.fromarray(npimg.astype("uint8"))
    out_img.putpalette(vocpallete)
    return out_img


def _getvocpallete(num_cls):
    n = num_cls
    pallete = [0] * (n * 3)
    for j in range(0, n):
        lab = j
        pallete[j * 3 + 0] = 0
        pallete[j * 3 + 1] = 0
        pallete[j * 3 + 2] = 0
        i = 0
        while lab > 0:
            pallete[j * 3 + 0] |= ((lab >> 0) & 1) << (7 - i)
            pallete[j * 3 + 1] |= ((lab >> 1) & 1) << (7 - i)
            pallete[j * 3 + 2] |= ((lab >> 2) & 1) << (7 - i)
            i = i + 1
            lab >>= 3
    return pallete


vocpallete = _getvocpallete(256)

adepallete = [
    0,
    0,
    0,
    120,
    120,
    120,
    180,
    120,
    120,
    6,
    230,
    230,
    80,
    50,
    50,
    4,
    200,
    3,
    120,
    120,
    80,
    140,
    140,
    140,
    204,
    5,
    255,
    230,
    230,
    230,
    4,
    250,
    7,
    224,
    5,
    255,
    235,
    255,
    7,
    150,
    5,
    61,
    120,
    120,
    70,
    8,
    255,
    51,
    255,
    6,
    82,
    143,
    255,
    140,
    204,
    255,
    4,
    255,
    51,
    7,
    204,
    70,
    3,
    0,
    102,
    200,
    61,
    230,
    250,
    255,
    6,
    51,
    11,
    102,
    255,
    255,
    7,
    71,
    255,
    9,
    224,
    9,
    7,
    230,
    220,
    220,
    220,
    255,
    9,
    92,
    112,
    9,
    255,
    8,
    255,
    214,
    7,
    255,
    224,
    255,
    184,
    6,
    10,
    255,
    71,
    255,
    41,
    10,
    7,
    255,
    255,
    224,
    255,
    8,
    102,
    8,
    255,
    255,
    61,
    6,
    255,
    194,
    7,
    255,
    122,
    8,
    0,
    255,
    20,
    255,
    8,
    41,
    255,
    5,
    153,
    6,
    51,
    255,
    235,
    12,
    255,
    160,
    150,
    20,
    0,
    163,
    255,
    140,
    140,
    140,
    250,
    10,
    15,
    20,
    255,
    0,
    31,
    255,
    0,
    255,
    31,
    0,
    255,
    224,
    0,
    153,
    255,
    0,
    0,
    0,
    255,
    255,
    71,
    0,
    0,
    235,
    255,
    0,
    173,
    255,
    31,
    0,
    255,
    11,
    200,
    200,
    255,
    82,
    0,
    0,
    255,
    245,
    0,
    61,
    255,
    0,
    255,
    112,
    0,
    255,
    133,
    255,
    0,
    0,
    255,
    163,
    0,
    255,
    102,
    0,
    194,
    255,
    0,
    0,
    143,
    255,
    51,
    255,
    0,
    0,
    82,
    255,
    0,
    255,
    41,
    0,
    255,
    173,
    10,
    0,
    255,
    173,
    255,
    0,
    0,
    255,
    153,
    255,
    92,
    0,
    255,
    0,
    255,
    255,
    0,
    245,
    255,
    0,
    102,
    255,
    173,
    0,
    255,
    0,
    20,
    255,
    184,
    184,
    0,
    31,
    255,
    0,
    255,
    61,
    0,
    71,
    255,
    255,
    0,
    204,
    0,
    255,
    194,
    0,
    255,
    82,
    0,
    10,
    255,
    0,
    112,
    255,
    51,
    0,
    255,
    0,
    194,
    255,
    0,
    122,
    255,
    0,
    255,
    163,
    255,
    153,
    0,
    0,
    255,
    10,
    255,
    112,
    0,
    143,
    255,
    0,
    82,
    0,
    255,
    163,
    255,
    0,
    255,
    235,
    0,
    8,
    184,
    170,
    133,
    0,
    255,
    0,
    255,
    92,
    184,
    0,
    255,
    255,
    0,
    31,
    0,
    184,
    255,
    0,
    214,
    255,
    255,
    0,
    112,
    92,
    255,
    0,
    0,
    224,
    255,
    112,
    224,
    255,
    70,
    184,
    160,
    163,
    0,
    255,
    153,
    0,
    255,
    71,
    255,
    0,
    255,
    0,
    163,
    255,
    204,
    0,
    255,
    0,
    143,
    0,
    255,
    235,
    133,
    255,
    0,
    255,
    0,
    235,
    245,
    0,
    255,
    255,
    0,
    122,
    255,
    245,
    0,
    10,
    190,
    212,
    214,
    255,
    0,
    0,
    204,
    255,
    20,
    0,
    255,
    255,
    255,
    0,
    0,
    153,
    255,
    0,
    41,
    255,
    0,
    255,
    204,
    41,
    0,
    255,
    41,
    255,
    0,
    173,
    0,
    255,
    0,
    245,
    255,
    71,
    0,
    255,
    122,
    0,
    255,
    0,
    255,
    184,
    0,
    92,
    255,
    184,
    255,
    0,
    0,
    133,
    255,
    255,
    214,
    0,
    25,
    194,
    194,
    102,
    255,
    0,
    92,
    0,
    255,
]

cityspallete = [
    128,
    64,
    128,
    244,
    35,
    232,
    70,
    70,
    70,
    102,
    102,
    156,
    190,
    153,
    153,
    153,
    153,
    153,
    250,
    170,
    30,
    220,
    220,
    0,
    107,
    142,
    35,
    152,
    251,
    152,
    0,
    130,
    180,
    220,
    20,
    60,
    255,
    0,
    0,
    0,
    0,
    142,
    0,
    0,
    70,
    0,
    60,
    100,
    0,
    80,
    100,
    0,
    0,
    230,
    119,
    11,
    32,
]

In [ ]:
from typing import Any
import PIL
import PIL.PngImagePlugin


class Trainer(object):
    def __init__(self, args):
        self.args = args
        # image transform
        input_transform = transforms.Compose(
            [
                transforms.ToImage(),  # Convert to tensor, only needed if you had a PIL image
                transforms.ToDtype(
                    torch.uint8, scale=True
                ),  # optional, most input are already uint8 at this point
                # ...
                transforms.RandomResizedCrop(
                    size=args.crop_size,
                ),  # Or Resize(antialias=True)
                # ...
                transforms.ToDtype(
                    torch.float32, scale=True
                ),  # Normalize expects float input
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ]
        )

        # NOTE: theres no default method to convert to a tensor
        def target_transport(x: PIL.PngImagePlugin.PngImageFile) -> Any:
            z = np.asarray(x, copy=True)
            n = torch.asarray(z, copy=True)
            return n

        dataset_cities_train = Cityscapes(
            "datasets/cityscapes/",
            split="train",
            mode="fine",
            target_type="semantic",
            transform=input_transform,
            target_transform=target_transport
        )
        dataset_cities_val = Cityscapes(
            "datasets/cityscapes",
            split="val",
            mode="fine",
            target_type="semantic",
            transform=input_transform,
            target_transform=target_transport
        )
        # dataset and dataloader
        data_kwargs = {
            "transform": input_transform,
            "base_size": args.base_size,
            "crop_size": args.crop_size,
        }
        train_dataset = dataset_cities_train
        val_dataset = dataset_cities_val
        self.train_loader: data.DataLoader[Cityscapes] = data.DataLoader(
            dataset=train_dataset,
            batch_size=args.batch_size,
            shuffle=True,
            drop_last=True,
        )
        self.val_loader = data.DataLoader(
            dataset=val_dataset, batch_size=1, shuffle=False
        )

        # create network
        self.model = get_fast_scnn(dataset=args.dataset, aux=args.aux)
        if torch.cuda.device_count() > 1:
            self.model = torch.nn.DataParallel(self.model, device_ids=[0, 1, 2])
        self.model.to(args.device)

        # resume checkpoint if needed
        if args.resume:
            if os.path.isfile(args.resume):
                name, ext = os.path.splitext(args.resume)
                assert (
                    ext == ".pkl" or ".pth"
                ), "Sorry only .pth and .pkl files supported."
                print("Resuming training, loading {}...".format(args.resume))
                self.model.load_state_dict(
                    torch.load(args.resume, map_location=lambda storage, loc: storage)
                )

        # create criterion
        self.criterion = MixSoftmaxCrossEntropyOHEMLoss(
            aux=args.aux, aux_weight=args.aux_weight, ignore_index=-1
        ).to(args.device)

        # optimizer
        self.optimizer = torch.optim.SGD(
            self.model.parameters(),
            lr=args.lr,
            momentum=args.momentum,
            weight_decay=args.weight_decay,
        )

        # lr scheduling
        self.lr_scheduler = LRScheduler(
            mode="poly",
            base_lr=args.lr,
            nepochs=args.epochs,
            iters_per_epoch=len(self.train_loader),
            power=0.9,
        )

        # evaluation metrics
        self.metric = SegmentationMetric(model.num_classes)

        self.best_pred = 0.0

    def train(self):
        cur_iters = 0
        start_time = time.time()
        for epoch in range(self.args.start_epoch, self.args.epochs):
            self.model.train()

            for i, (images, targets) in enumerate(self.train_loader):
                cur_lr = self.lr_scheduler(cur_iters)
                for param_group in self.optimizer.param_groups:
                    param_group["lr"] = cur_lr

                images = images.to(self.args.device)
                targets = targets.to(self.args.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, targets)

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                cur_iters += 1
                if cur_iters % 10 == 0:
                    print(
                        "Epoch: [%2d/%2d] Iter [%4d/%4d] || Time: %4.4f sec || lr: %.8f || Loss: %.4f"
                        % (
                            epoch,
                            self.args.epochs,
                            i + 1,
                            len(self.train_loader),
                            time.time() - start_time,
                            cur_lr,
                            loss.item(),
                        )
                    )

            if self.args.no_val:
                # save every epoch
                save_checkpoint(self.model, self.args, is_best=False)
            else:
                self.validation(epoch)

        save_checkpoint(self.model, self.args, is_best=False)

    def validation(self, epoch):
        is_best = False
        self.metric.reset()
        self.model.eval()
        for i, (image, target) in enumerate(self.val_loader):
            image = image.to(self.args.device)

            outputs = self.model(image)
            pred = torch.argmax(outputs[0], 1)
            pred = pred.cpu().data.numpy()
            self.metric.update(pred, target.numpy())
            pixAcc, mIoU = self.metric.get()
            print(
                "Epoch %d, Sample %d, validation pixAcc: %.3f%%, mIoU: %.3f%%"
                % (epoch, i + 1, pixAcc * 100, mIoU * 100)
            )

        new_pred = (pixAcc + mIoU) / 2
        if new_pred > self.best_pred:
            is_best = True
            self.best_pred = new_pred
        save_checkpoint(self.model, self.args, is_best)


def save_checkpoint(model, args, is_best=False):
    """Save Checkpoint"""
    directory = os.path.expanduser(args.save_folder)
    if not os.path.exists(directory):
        os.makedirs(directory)
    filename = "{}_{}.pth".format(args.model, args.dataset)
    save_path = os.path.join(directory, filename)
    torch.save(model.state_dict(), save_path)
    if is_best:
        best_filename = "{}_{}_best_model.pth".format(args.model, args.dataset)
        best_filename = os.path.join(directory, best_filename)
        shutil.copyfile(filename, best_filename)

In [ ]:
class ModelArgs:
  def __init__(self) -> None:
    self.model = "fast_scnn"
    self.dataset = "citys"
    self.base_size = 1024
    self.crop_size = 768
    self.train_split = "train"
    
    # Aux loss
    self.aux = False
    self.aux_weight = 0.4
    self.epochs = 160
    self.start_epoch = 0
    self.batch_size = 2
    self.lr = 1e-2
    self.momentum = 0.9
    self.weight_decay = 1e-4
    self.save_folder = "./weights"
    self.resume: str = ""
    self.is_eval = False
    self.skip_val = True
    self.device = torch.device("cpu")

In [ ]:
args = ModelArgs()

trainer = Trainer(args)
if args.is_eval:
    print("Evaluation model: ", args.resume)
    trainer.validation(args.start_epoch)
else:
    print("Starting Epoch: %d, Total Epochs: %d" % (args.start_epoch, args.epochs))
    trainer.train()

w/ class balance
Starting Epoch: 0, Total Epochs: 160


AttributeError: 'NoneType' object has no attribute 'requires_grad'